In [ ]:
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-Bench_Verified", split="test")

In [ ]:
import csv

GEMINI_3_METADATA_PATH = "../../datasets/gemini-3-pro-preview-20251118.csv"
GPT_52_METADATA_PATH = "../../datasets/gpt-5.2-20251211.csv"
CLAUDE_45_METADATA_PATH = "../../datasets/claude-sonnet-4.5.csv"
GPT_51_CODEX_METADATA_PATH = "../../datasets/gpt-51-codex.csv"
KIMI_METADATA_PATH = "../../datasets/kimi-k2.csv"
DEEPSEEK_METADATA_PATH = "../../datasets/deepseek.csv"

with open(GEMINI_3_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    gemini_metadata = list(reader)

with open(GPT_52_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    gpt_metadata = list(reader)

with open(CLAUDE_45_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    claude_metadata = list(reader)

with open(GPT_51_CODEX_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    gpt_codex_metadata = list(reader)

with open(KIMI_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    kimi_metadata = list(reader)

with open(DEEPSEEK_METADATA_PATH, "r") as f:
    reader = csv.DictReader(f)
    deepseek_metadata = list(reader)

print(f"Gemini-3 metadata loaded: {len(gemini_metadata)}")
print(f"GPT-5.2 metadata loaded: {len(gpt_metadata)}")
print(f"Claude-4.5 metadata loaded: {len(claude_metadata)}")
print(f"GPT-5.1 Codex metadata loaded: {len(gpt_codex_metadata)}")
print(f"Kimi metadata loaded: {len(kimi_metadata)}")
print(f"DeepSeek metadata loaded: {len(deepseek_metadata)}")

gemini_metadata[0]

In [ ]:

import re

def clean_messages(messages):
    cleaned_messages = []
    
    for message in messages:
        if message.get("role") == "tool":
            pattern = r"^Exit code: (\d+)\n\n([\s\S]*)$"
            match = re.match(pattern, message.get("content"))
            if match:
                exit_code = match.group(1)
                content = match.group(2)
                cleaned_messages.append({
                    "role": "user",
                    "content": f"<returncode>{exit_code}</returncode>\n<output>\n{content}\n</output>"
                })
        elif message.get("role") == "assistant":
            if message.get("tool_calls") is None:
                print("No tool calls")
                cleaned_messages.append({
                    "role": "assistant",
                    "content": message.get("content", "")
                })
            elif len(message.get("tool_calls", [])) == 1:
                cleaned_messages.append({
                    "role": "assistant",
                    "content": message.get("content", "").strip() + "\n\n" + message.get("tool_calls")[0].get("view").get("content")
                })
            else:
                print("Multiple tool calls")
        else:
            # if message.get("role") == "user" and message.get('content').startswith("Please always provide EXACTLY ONE action in triple backticks"):
            #     print("Format error! skipped")
            #     continue
            cleaned_messages.append({
                "role": message.get("role"),
                "content": message.get("content")
            })
        
    return cleaned_messages

In [ ]:
import json
import requests
from tqdm import tqdm

num_sample_from_gemini = 0
num_sample_from_gpt = 0
num_sample_from_claude = 0
num_sample_from_gpt_codex = 0
num_sample_from_kimi = 0
num_sample_from_deepseek = 0

gemini_ids = []
gpt_ids = []
claude_ids = []
gpt_codex_ids = []
kimi_ids = []
deepseek_ids = []
no_resolved_ids = []

# ========== HTTP request config ==========
url_template = "https://api.docent.transluce.org/rest/{}/agent_run_with_tree"

headers = {
    "Cookie": "docent_session=6370cfd8-531c-48c2-b542-1658ffcbafa3; ph_phc_Xvh5cw8HEyhTOOnZWj3bT00NeCPXp2fswJzGvrR5d4w_posthog=%7B%22distinct_id%22%3A%22019b2574-9516-719e-a29c-e4e16e90e6c6%22%2C%22%24sesid%22%3A%5B1765871481356%2C%22019b2607-2d60-7a92-a590-a40b72b12e78%22%2C1765869563232%5D%2C%22%24initial_person_info%22%3A%7B%22r%22%3A%22%24direct%22%2C%22u%22%3A%22https%3A%2F%2Fdocent.transluce.org%2Fdashboard%2F6f6e2390-00c7-479f-9a2a-87b53b7477cb%22%7D%7D"
}

params = {
    "apply_base_where_clause": "false",
    "full_tree": "false"
}

for i in tqdm(range(500)):
    file_path = f"../../datasets/train/swe/swe_{i}.json"
    assert gemini_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]
    assert gpt_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]
    assert gpt_codex_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]
    assert kimi_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]
    assert deepseek_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]
    # assert claude_metadata[i]["metadata.instance_id"] == dataset[i]["instance_id"]

    # Gemini    
    if gemini_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_gemini += 1
        gemini_ids.append(i)
        params['agent_run_id'] = gemini_metadata[i].get("agent_run_id")
        url = url_template.format("3641b17f-034e-4b36-aa66-471dfed837d6")
        model_name = "Gemini 3 Pro Preview"

    # GPT
    elif gpt_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_gpt += 1
        gpt_ids.append(i)
        params['agent_run_id'] = gpt_metadata[i].get("agent_run_id")
        url = url_template.format("93ef4179-a475-4f0b-88d1-46b5aa75e19b")
        model_name = "GPT-5.2 (2025-12-11)"
        
    # Claude
    elif claude_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_claude += 1
        claude_ids.append(i)
        params['agent_run_id'] = claude_metadata[i].get("agent_run_id")
        url = url_template.format("a4844da1-fbb9-4d61-b82c-f46e471f748a")
        model_name = "Claude 4.5 Sonnet"
    
    # GPT 5.1 Codex (medium reasoning)   
    elif gpt_codex_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_gpt_codex += 1
        gpt_codex_ids.append(i)
        params['agent_run_id'] = gpt_codex_metadata[i].get('agent_run_id')
        url = url_template.format("ebeb5896-b0bc-4504-a687-a5bf9824a146")
        model_name = "GPT-5.1 Codex"
    
    # Kimi K2 thinking
    elif kimi_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_kimi += 1
        kimi_ids.append(i)
        params['agent_run_id'] = kimi_metadata[i].get('agent_run_id')
        url = url_template.format("9dcb7016-5356-4966-a184-ce4b1131f33c")
        model_name = "Kimi K2"
    
    # Deepseek V3.2 Reasoner 
    elif deepseek_metadata[i]['metadata.scores.resolved'] == "1":
        num_sample_from_deepseek += 1
        deepseek_ids.append(i)
        params['agent_run_id'] = deepseek_metadata[i].get('agent_run_id')
        url = url_template.format("7fde5552-6b17-4cb7-ab9c-15fd9fb5b845")
        model_name = "DeepSeek V3.2 Reasoner"
    
    else:
        print(f"Sample {i} not resolved by any model")
        no_resolved_ids.append(i)
        continue
    
    # Send HTTP Request
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        try:
            data = response.json()
            cleaned_messages = clean_messages(data[0].get("transcripts")[0].get("messages"))            
            with open (file_path, "w") as f:
                json.dump({
                    "task_id": i,
                    "conversations": cleaned_messages,
                    "metadata": {
                        "model": model_name,
                        "url": url, 
                        "agent_run_id": params['agent_run_id']
                    }
                }, f, ensure_ascii=False, indent=4)
        except requests.exceptions.JSONDecodeError:
            print("Response is not JSON:")
            print(response.text)
    else:
        print(f"Request failed: {response.text}")
    
total_samples = num_sample_from_gemini + num_sample_from_gpt + num_sample_from_claude + num_sample_from_gpt_codex + num_sample_from_kimi + num_sample_from_deepseek    
print(f"Sampled from Gemini: {num_sample_from_gemini}")
print(f"Sampled from GPT: {num_sample_from_gpt}")
print(f"Sampled from Claude: {num_sample_from_claude}")  
print(f"Sampled from GPT Codex: {num_sample_from_gpt_codex}")
print(f"Sampled from Kimi: {num_sample_from_kimi}")
print(f"Sampled from DeepSeek: {num_sample_from_deepseek}")
print(f"Total: {total_samples}") 


with open("../../datasets/train/swe/ids.json", "w") as f:
    json.dump({
        "total": total_samples,
        "gemini": gemini_ids,
        "gpt": gpt_ids,
        "claude": claude_ids,
        "gpt_codex": gpt_codex_ids,
        "kimi": kimi_ids,
        "deepseek": deepseek_ids,
        "no_resolved": no_resolved_ids
    }, f, ensure_ascii=False, indent=4)
